# Batch MAELIA — Plan d'Expérience SMT sur terrainSA

Ce notebook lance un plan d'expérience SMT sur `terrainSA`, où la parcelle `beauce_5_1`
est clonée 100 fois sous le même îlot `beauce_5`. Chaque clone reçoit un point DOE
différent, ce qui amortit l'initialisation GAMA sur plusieurs points de sensibilité.

**Suivi des logs en temps réel** — ouvrir un terminal et lancer :
```bash
tail -f /tmp/gama_smt_terrainSA.log
```


## 0. Imports & configuration

In [1]:
import os, sys, subprocess, time, io, warnings, shutil
from pathlib import Path
from datetime import date, timedelta
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# SMT
from smt.design_space import FloatVariable, OrdinalVariable, CategoricalVariable
from smt_design_space_ext import AdsgDesignSpaceImpl

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'savefig.bbox': 'tight'})

# ─── Chemins ──────────────────────────────────────────────────────────────────
GAMA_HEADLESS = '/Applications/Gama.app/Contents/headless/gama-headless.sh'
MAELIA_ROOT   = Path('/Users/benjamin/Workspace_GAMA/MAELIA')
MODEL_PATH    = MAELIA_ROOT / 'models/main/launcherBase.gaml'
LOG_DIR       = MAELIA_ROOT / 'models/main/log'
NOM_DECOUPAGE = 'terrainSA'
VARIANTS_DIR  = MAELIA_ROOT / f'includes/{NOM_DECOUPAGE}/modeleAgricole/agriculteurs/variants_SMT'
BLOC_DONNEES  = MAELIA_ROOT / f'includes/{NOM_DECOUPAGE}/modeleAgricole/blocsDonnees.csv'
XML_DIR       = Path('/tmp/maelia_smt_terrainSA_xml')
GAMA_LOG      = Path('/tmp/gama_smt_terrainSA.log')

# ─── Paramètres de simulation ─────────────────────────────────────────────────
ANNEE_DEBUT   = 2019
NB_ANNEES     = 12
ANNEES_DOSE   = list(range(2019, 2031))
FINAL_STEP    = 1200
CROP          = 'ble'
CHEMIN_RACINE = str(MAELIA_ROOT) + '/'

# ─── Plan d'expérience groupé ─────────────────────────────────────────────────
TARGET_N_DOE = 10000
CLONES_PER_RUN = 100

_all_parcs = []
with open(BLOC_DONNEES) as _f:
    for _line in _f:
        _parts = [_p.strip() for _p in _line.strip().split(';') if _p.strip()]
        if len(_parts) >= 2:
            _all_parcs.extend(_parts[1:])

ALL_PARCELLES = [p for p in _all_parcs if p.startswith('beauce_5_sa_')]
ALL_PARCELLES = sorted(ALL_PARCELLES)[:CLONES_PER_RUN]
if len(ALL_PARCELLES) != CLONES_PER_RUN:
    raise ValueError(f'terrainSA doit contenir {CLONES_PER_RUN} clones, trouvé {len(ALL_PARCELLES)}')

N_SIMU = int(np.ceil(TARGET_N_DOE / len(ALL_PARCELLES)))
N_DOE  = N_SIMU * len(ALL_PARCELLES)

# ─── Création des dossiers ────────────────────────────────────────────────────
XML_DIR.mkdir(parents=True, exist_ok=True)
VARIANTS_DIR.mkdir(parents=True, exist_ok=True)

# ─── Nettoyage des anciens fichiers SMT terrainSA ─────────────────────────────
old_dirs = sorted(LOG_DIR.glob(f'{NOM_DECOUPAGE}_smt_*'))
if old_dirs:
    print(f'Nettoyage de {len(old_dirs)} anciens répertoire(s) de sortie SMT terrainSA...')
    for d in old_dirs:
        shutil.rmtree(d)
    print('  OK.')

old_csvs = sorted(VARIANTS_DIR.glob('dateDose_smt_*.csv'))
if old_csvs:
    print(f'Nettoyage de {len(old_csvs)} anciens fichier(s) dateDose SMT terrainSA...')
    for f in old_csvs:
        f.unlink()
    print('  OK.')

print('Configuration OK')
print(f'  Terrain       : {NOM_DECOUPAGE}')
print(f'  Culture       : {CROP}')
print(f'  Campagnes     : {ANNEES_DOSE[0]}–{ANNEES_DOSE[-1]}')
print(f'  Simulations   : {N_SIMU} runs')
print(f'  Clones/run    : {len(ALL_PARCELLES)}')
print(f'  Points DOE    : {N_DOE}')
print()
print('─' * 55)
print('Pour suivre les logs GAMA dans un terminal :')
print(f'  tail -f {GAMA_LOG}')
print('─' * 55)


Configuration OK
  Terrain       : terrainSA
  Culture       : ble
  Campagnes     : 2019–2030
  Simulations   : 100 runs
  Clones/run    : 100
  Points DOE    : 10000

───────────────────────────────────────────────────────
Pour suivre les logs GAMA dans un terminal :
  tail -f /tmp/gama_smt_terrainSA.log
───────────────────────────────────────────────────────


## 1. Espace de conception SMT

Reprise de l'espace de conception hiérarchique défini dans `Try_MAELIA.ipynb` (26 variables : 14 discrètes/catégorielles + 12 continues).

In [2]:
# ─── Types de préparation et de fertilisation ─────────────────────────────────
prepa_types = ['Labour', 'Preparation du lit de semence', 'Déchaumage', 'Roulage']
ferti_types = ['N_solution', 'AN', 'mineral', 'urea']

# Notes sur le calibrage des variables continues pour le blé d'hiver :
#   - Jour_Semis        : 260–320  (semis automnal mi-sept → mi-nov)
#   - Jours_Semis_avant_PREPA : 1–30 (préparation du sol 0-30 j avant semis)
#   - Jours_Semis_vers_F1 : 100–250 (ferti printanière 100-250 j après semis)
#   - Jours_F1_vers_F2  : 1–60   (2e apport printanier 2-8 semaines après le 1er)
#   - Jours_F2_vers_F3  : 1–60   (idem)
#   - Jours_dernière_op_vers_Récolte : 1–200 (récolte estivale)

agri_design_space = AdsgDesignSpaceImpl(
    design_variables=[
        # --- Bloc discret (tags uniques) ---
        OrdinalVariable(['0_ferti', '1_ferti', '2_ferti', '3_ferti']),  # 0: n_ferti
        OrdinalVariable(['Non_prepa', 'Oui_prepa']),                    # 1: has_prepa
        OrdinalVariable(['1_prepa', '2_prepa']),                        # 2: nb_prepa
        CategoricalVariable([p + '_p1' for p in prepa_types]),          # 3: prepa_1
        CategoricalVariable([p + '_p2' for p in prepa_types]),          # 4: prepa_2

        OrdinalVariable(['1_f1', '2_f1']),                              # 5: Nb produits F1
        CategoricalVariable([f + '_f11' for f in ferti_types]),         # 6: type F1_1
        CategoricalVariable([f + '_f12' for f in ferti_types]),         # 7: type F1_2

        OrdinalVariable(['1_f2', '2_f2']),                              # 8: Nb produits F2
        CategoricalVariable([f + '_f21' for f in ferti_types]),         # 9: type F2_1
        CategoricalVariable([f + '_f22' for f in ferti_types]),         # 10: type F2_2

        OrdinalVariable(['1_f3', '2_f3']),                              # 11: Nb produits F3
        CategoricalVariable([f + '_f31' for f in ferti_types]),         # 12: type F3_1
        CategoricalVariable([f + '_f32' for f in ferti_types]),         # 13: type F3_2

        # --- Bloc continu -----------------------------------------------
        FloatVariable(260, 320),   # 14: Jour_Semis (semis automnal, blé d'hiver)
        FloatVariable(1,   30),    # 15: Jours_Semis_avant_PREPA
        FloatVariable(100, 250),   # 16: Jours_Semis_vers_F1 (ferti printanière)
        FloatVariable(1,   60),    # 17: Jours_F1_vers_F2
        FloatVariable(1,   60),    # 18: Jours_F2_vers_F3
        FloatVariable(1,   200),   # 19: Jours_dernière_op_vers_Récolte
        FloatVariable(10,  100),   # 20: dose F1_1
        FloatVariable(10,  100),   # 21: dose F1_2
        FloatVariable(10,  100),   # 22: dose F2_1
        FloatVariable(10,  100),   # 23: dose F2_2
        FloatVariable(10,  100),   # 24: dose F3_1
        FloatVariable(10,  100),   # 25: dose F3_2
    ]
)

# ─── Hiérarchie ADSG ─────────────────────────────────────────────────────────
agri_design_space.declare_decreed_var(decreed_var=2,  meta_var=1, meta_value=['Oui_prepa'])
agri_design_space.declare_decreed_var(decreed_var=3,  meta_var=1, meta_value=['Oui_prepa'])
agri_design_space.declare_decreed_var(decreed_var=15, meta_var=1, meta_value=['Oui_prepa'])
agri_design_space.declare_decreed_var(decreed_var=4,  meta_var=2, meta_value=['2_prepa'])

agri_design_space.declare_decreed_var(decreed_var=16, meta_var=0, meta_value=['1_ferti', '2_ferti', '3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=5,  meta_var=0, meta_value=['1_ferti', '2_ferti', '3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=6,  meta_var=0, meta_value=['1_ferti', '2_ferti', '3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=20, meta_var=0, meta_value=['1_ferti', '2_ferti', '3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=7,  meta_var=5, meta_value=['2_f1'])
agri_design_space.declare_decreed_var(decreed_var=21, meta_var=5, meta_value=['2_f1'])

agri_design_space.declare_decreed_var(decreed_var=17, meta_var=0, meta_value=['2_ferti', '3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=8,  meta_var=0, meta_value=['2_ferti', '3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=9,  meta_var=0, meta_value=['2_ferti', '3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=22, meta_var=0, meta_value=['2_ferti', '3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=10, meta_var=8, meta_value=['2_f2'])
agri_design_space.declare_decreed_var(decreed_var=23, meta_var=8, meta_value=['2_f2'])

agri_design_space.declare_decreed_var(decreed_var=18, meta_var=0, meta_value=['3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=11, meta_var=0, meta_value=['3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=12, meta_var=0, meta_value=['3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=24, meta_var=0, meta_value=['3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=13, meta_var=11, meta_value=['2_f3'])
agri_design_space.declare_decreed_var(decreed_var=25, meta_var=11, meta_value=['2_f3'])

print('Espace de conception SMT défini :', len(agri_design_space.design_variables), 'variables')
print('  Jour_Semis      : 260–320  (semis automnal blé d\'hiver)')
print('  Jours_vers_F1   : 100–250  (ferti printanière)')
print('  Jours_F1_vs_F2/3: 1–60    (inter-apports printaniers)')


# ─── Fonctions de décodage (issues de Try_MAELIA.ipynb) ──────────────────────

def parse_smt_row(row_data):
    """Analyse une ligne décodée par SMT en cherchant les tags uniques."""
    parsed = {
        'n_ferti': 0, 'has_prepa': False, 'nb_prepa': 1,
        'nb_f1': 1, 'nb_f2': 1, 'nb_f3': 1,
        'prepa_t1': '-', 'prepa_t2': '-',
        'f1_t1': '-', 'f1_t2': '-',
        'f2_t1': '-', 'f2_t2': '-',
        'f3_t1': '-', 'f3_t2': '-',
        'floats': []
    }
    values = list(row_data.values()) if isinstance(row_data, dict) else row_data
    for val in values:
        if isinstance(val, (float, int)):
            parsed['floats'].append(float(val))
        elif isinstance(val, str):
            if '_ferti' in val:          parsed['n_ferti']  = int(val[0])
            elif val in ('Oui_prepa', 'Non_prepa'): parsed['has_prepa'] = (val == 'Oui_prepa')
            elif '_prepa' in val:        parsed['nb_prepa'] = int(val[0])
            elif '_p1' in val:           parsed['prepa_t1'] = val.replace('_p1', '')
            elif '_p2' in val:           parsed['prepa_t2'] = val.replace('_p2', '')
            elif '_f11' in val:          parsed['f1_t1']    = val.replace('_f11', '')
            elif '_f12' in val:          parsed['f1_t2']    = val.replace('_f12', '')
            elif val in ('1_f1', '2_f1'): parsed['nb_f1']   = int(val[0])
            elif '_f21' in val:          parsed['f2_t1']    = val.replace('_f21', '')
            elif '_f22' in val:          parsed['f2_t2']    = val.replace('_f22', '')
            elif val in ('1_f2', '2_f2'): parsed['nb_f2']   = int(val[0])
            elif '_f31' in val:          parsed['f3_t1']    = val.replace('_f31', '')
            elif '_f32' in val:          parsed['f3_t2']    = val.replace('_f32', '')
            elif val in ('1_f3', '2_f3'): parsed['nb_f3']   = int(val[0])
    return parsed


def extract_table_from_smt(x_array, dsg_space):
    """Construit un DataFrame lisible depuis la matrice DOE."""
    results = []
    decoded = dsg_space.decode_values(x_array)
    for row_data in decoded:
        p  = parse_smt_row(row_data)
        fl = p['floats']
        while len(fl) < 12:
            fl.append(float('nan'))

        semi = int(round(fl[0]))
        prepa_date = int(round(max(1, semi - fl[1]))) if p['has_prepa'] else None
        f1_date = int(round(semi + fl[2])) if p['n_ferti'] >= 1 else None
        f2_date = int(round(f1_date + fl[3])) if p['n_ferti'] >= 2 and f1_date else None
        f3_date = int(round(f2_date + fl[4])) if p['n_ferti'] >= 3 and f2_date else None
        last    = f3_date or f2_date or f1_date or (prepa_date or semi)
        recolte = int(round(last + fl[5]))

        def _d(v): return v if v else '-'
        def _r(v): return round(float(v), 1) if v == v else '-'

        results.append({
            'n_ferti':    p['n_ferti'],
            'has_prepa':  'Oui' if p['has_prepa'] else 'Non',
            'Semis_jj':   semi,
            'PREPA_jj':   _d(prepa_date),
            'PREPA_T1':   p['prepa_t1'],
            'PREPA_T2':   p['prepa_t2'] if p['nb_prepa'] == 2 else '-',
            'F1_jj':      _d(f1_date),
            'F1_T1':      p['f1_t1'],   'F1_D1': _r(fl[6]),
            'F1_T2':      p['f1_t2'] if p['nb_f1'] == 2 else '-',
            'F1_D2':      _r(fl[7]) if p['nb_f1'] == 2 else '-',
            'F2_jj':      _d(f2_date),
            'F2_T1':      p['f2_t1'],   'F2_D1': _r(fl[8]),
            'F3_jj':      _d(f3_date),
            'F3_T1':      p['f3_t1'],   'F3_D1': _r(fl[10]),
            'Recolte_jj': recolte,
        })
    return pd.DataFrame(results)

Espace de conception SMT défini : 26 variables
  Jour_Semis      : 260–320  (semis automnal blé d'hiver)
  Jours_vers_F1   : 100–250  (ferti printanière)
  Jours_F1_vs_F2/3: 1–60    (inter-apports printaniers)


## 2. Génération du plan d'expérience (DOE)

In [3]:
print(f'Génération de {N_DOE} points LHS hiérarchiques (SMT ADSG)...')
result = agri_design_space._sample_valid_x(N_DOE)
xt, is_acting_t = result[0], result[1]
print(f'Matrice DOE : {xt.shape[0]} × {xt.shape[1]} variables')
print()

doe_display = extract_table_from_smt(xt, agri_design_space)
print(f'Aperçu du plan d\'expérience (N={N_DOE}) :')
display(doe_display)

Génération de 10000 points LHS hiérarchiques (SMT ADSG)...
Matrice DOE : 10000 × 26 variables

Aperçu du plan d'expérience (N=10000) :


,n_ferti,has_prepa,Semis_jj,PREPA_jj,PREPA_T1,PREPA_T2,F1_jj,F1_T1,F1_D1,F1_T2,F1_D2,F2_jj,F2_T1,F2_D1,F3_jj,F3_T1,F3_D1,Recolte_jj
0,1,Non,310,-,Labour,-,499,N_solution,54.5,-,-,-,N_solution,55.0,-,N_solution,55.0,520
1,0,Oui,282,279,Roulage,Déchaumage,-,N_solution,55.0,-,-,-,N_solution,55.0,-,N_solution,55.0,292
2,3,Non,290,-,Labour,-,407,N_solution,19.6,-,-,436,mineral,67.5,465,N_solution,86.9,544
3,2,Non,267,-,Labour,-,423,N_solution,87.6,-,-,431,N_solution,54.2,-,N_solution,55.0,493
4,2,Non,265,-,Labour,-,365,AN,27.3,-,-,367,N_solution,80.2,-,N_solution,55.0,531
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0,Non,275,-,Labour,-,-,N_solution,55.0,-,-,-,N_solution,55.0,-,N_solution,55.0,420
9996,3,Non,273,-,Labour,-,415,urea,27.8,-,-,471,urea,39.4,506,N_solution,87.3,623
9997,0,Oui,301,300,Déchaumage,Déchaumage,-,N_solution,55.0,-,-,-,N_solution,55.0,-,N_solution,55.0,394
9998,0,Non,273,-,Labour,-,-,N_solution,55.0,-,-,-,N_solution,55.0,-,N_solution,55.0,284


## 3. Génération des fichiers `dateDose.csv`

Chaque point du DOE est converti en fichier `dateDose_smt_XXX.csv` au format MAELIA :
- **PREPA** : `fl[1]` jours **avant** le semis (préparation du sol).
- **SEMIS** : jour `fl[0]` de l'année.
- **FERTI_Min** : `fl[2]` jours après semis (F1), puis `fl[3]` jours pour F2, `fl[4]` pour F3.
- **RECOLTE** : `fl[5]` jours après la dernière opération.

In [4]:
PREPA_DEPTHS = {
    'Labour': '24',
    'Preparation du lit de semence': '10',
    'Déchaumage': '',
    'Roulage': '',
}

COLS = ['Trait', 'nom_MAELIA', 'plant_sem', 'annee_deb',
        'id_operation', 'ordre_op', 'DATE', 'PROF', 'TYPE', 'DOSE']

MIN_GROWING_DAYS = 200
HARVEST_BUFFER_DAYS = 7
MIN_DAYS_LAST_OP_TO_HARVEST = 1

def days_in_year(year):
    return 366 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 365

def doy_to_date_str(year, doy):
    return (date(year, 1, 1) + timedelta(days=int(doy) - 1)).strftime('%d/%m/%Y')

def parse_date_str(value):
    return pd.to_datetime(value, dayfirst=True).date()

def safe_f(val, default=50.0):
    try:
        v = float(val)
        return default if v != v else v
    except (TypeError, ValueError):
        return default

def validate_datedose_rows(rows):
    """Détecte les calendriers qui feraient planter agriculteurDateDose.gaml."""
    order = {'RECOLTE': 0, 'PREPA': 1, 'SEMIS': 2, 'FERTI_Min': 3}
    by_parcelle = {}
    issues = []
    for row in rows:
        by_parcelle.setdefault(row['Trait'], []).append(row)

    for parcelle_id, parcelle_ops in by_parcelle.items():
        culture_en_place = False
        sorted_ops = sorted(
            parcelle_ops,
            key=lambda row: (parse_date_str(row['DATE']),
                             order.get(row['id_operation'], 99),
                             str(row.get('ordre_op', ''))),
        )
        for row in sorted_ops:
            op = row['id_operation']
            day = row['DATE']
            campagne = row['annee_deb']
            if op == 'SEMIS':
                if culture_en_place:
                    issues.append(f'{parcelle_id} campagne {campagne}: SEMIS le {day} alors qu’une culture est déjà en place')
                culture_en_place = True
            elif op == 'RECOLTE':
                if not culture_en_place:
                    issues.append(f'{parcelle_id} campagne {campagne}: RECOLTE le {day} sans culture en place')
                culture_en_place = False
            elif op == 'FERTI_Min':
                if not culture_en_place:
                    issues.append(f'{parcelle_id} campagne {campagne}: FERTI_Min le {day} sans culture en place')
            elif op == 'PREPA':
                pass
            else:
                issues.append(f'{parcelle_id} campagne {campagne}: opération inconnue {op}')
    return issues

def generate_ops_for_parcelle(point_idx, parcelle_id, years):
    """Génère les opérations pour un SEUL point DOE appliqué à une SEULE parcelle."""
    decoded = agri_design_space.decode_values(xt[point_idx:point_idx + 1])
    p  = parse_smt_row(decoded[0])
    fl = [safe_f(v) for v in p['floats']]
    while len(fl) < 12: fl.append(50.0)

    ops = []
    for year in years:
        semi_doy  = max(1, int(round(fl[0])))
        prepa_doy = max(1, semi_doy - int(round(fl[1]))) if p['has_prepa'] else None
        f1_doy    = int(round(semi_doy + fl[2])) if p['n_ferti'] >= 1 else None
        f2_doy    = int(round(f1_doy  + fl[3])) if p['n_ferti'] >= 2 and f1_doy else None
        f3_doy    = int(round(f2_doy  + fl[4])) if p['n_ferti'] >= 3 and f2_doy else None

        next_entry_doy = prepa_doy if prepa_doy is not None else semi_doy
        latest_rec_doy = days_in_year(year) + next_entry_doy - HARVEST_BUFFER_DAYS
        max_last_op_doy = latest_rec_doy - MIN_DAYS_LAST_OP_TO_HARVEST

        ferti_events = []
        if f1_doy: ferti_events.append((f1_doy, p['nb_f1'], p['f1_t1'], fl[6],  p['f1_t2'], fl[7]))
        if f2_doy: ferti_events.append((f2_doy, p['nb_f2'], p['f2_t1'], fl[8],  p['f2_t2'], fl[9]))
        if f3_doy: ferti_events.append((f3_doy, p['nb_f3'], p['f3_t1'], fl[10], p['f3_t2'], fl[11]))
        ferti_events = [event for event in ferti_events if event[0] <= max_last_op_doy]

        last_doy = ferti_events[-1][0] if ferti_events else semi_doy
        wanted_rec_doy = max(semi_doy + MIN_GROWING_DAYS, int(round(last_doy + fl[5])))
        rec_doy = min(wanted_rec_doy, latest_rec_doy)
        if rec_doy <= last_doy:
            raise ValueError(
                f'Calendrier impossible pour {parcelle_id} campagne {year}: '
                f'dernière opération DOY {last_doy}, récolte max DOY {latest_rec_doy}'
            )

        base = {'Trait': parcelle_id, 'nom_MAELIA': f'P1_{year}',
                'plant_sem': CROP, 'annee_deb': year}

        if p['has_prepa']:
            ops.append({**base, 'id_operation': 'PREPA', 'ordre_op': '',
                        'DATE': doy_to_date_str(year, prepa_doy),
                        'PROF': PREPA_DEPTHS.get(p['prepa_t1'], ''), 'TYPE': p['prepa_t1'], 'DOSE': ''})
            if p['nb_prepa'] == 2:
                ops.append({**base, 'id_operation': 'PREPA', 'ordre_op': '2',
                            'DATE': doy_to_date_str(year, prepa_doy),
                            'PROF': PREPA_DEPTHS.get(p['prepa_t2'], ''), 'TYPE': p['prepa_t2'], 'DOSE': ''})

        ops.append({**base, 'id_operation': 'SEMIS', 'ordre_op': '',
                    'DATE': doy_to_date_str(year, semi_doy),
                    'PROF': '', 'TYPE': '', 'DOSE': ''})

        for ordre, (doy, nb, t1, d1, t2, d2) in enumerate(ferti_events, 1):
            ds = doy_to_date_str(year, doy)
            ops.append({**base, 'id_operation': 'FERTI_Min', 'ordre_op': str(ordre),
                        'DATE': ds, 'PROF': '', 'TYPE': t1, 'DOSE': str(round(d1, 1))})
            if nb == 2:
                ops.append({**base, 'id_operation': 'FERTI_Min', 'ordre_op': str(ordre),
                            'DATE': ds, 'PROF': '', 'TYPE': t2, 'DOSE': str(round(d2, 1))})

        ops.append({**base, 'id_operation': 'RECOLTE', 'ordre_op': '',
                    'DATE': doy_to_date_str(year, rec_doy),
                    'PROF': '', 'TYPE': '', 'DOSE': ''})
    return ops

# ─── Générer tous les fichiers dateDose (N_SIMU fichiers, CLONES_PER_RUN parcelles/fichier) ───
print(f'Génération de {N_SIMU} fichiers dateDose distribués...')
for i in range(N_SIMU):
    path = VARIANTS_DIR / f'dateDose_smt_{i:03d}.csv'
    all_ops = []
    for j, parcelle_id in enumerate(ALL_PARCELLES):
        point_idx = i * len(ALL_PARCELLES) + j
        all_ops.extend(generate_ops_for_parcelle(point_idx, parcelle_id, ANNEES_DOSE))

    issues = validate_datedose_rows(all_ops)
    if issues:
        preview = '\n  - '.join(issues[:10])
        raise ValueError(f'Calendrier dateDose invalide pour simulation {i:03d}:\n  - {preview}')

    df = pd.DataFrame(all_ops, columns=COLS)
    df.to_csv(path, sep=';', index=False)
    if i % 10 == 0 or i == N_SIMU - 1:
        print(f'  Simulation {i:03d} : {len(df)} opérations ({len(ALL_PARCELLES)} parcelles traitées)')

print(f'\nFichiers écrits dans : {VARIANTS_DIR}')


Génération de 100 fichiers dateDose distribués...
  Simulation 000 : 5904 opérations (100 parcelles traitées)
  Simulation 010 : 5544 opérations (100 parcelles traitées)
  Simulation 020 : 5628 opérations (100 parcelles traitées)
  Simulation 030 : 5892 opérations (100 parcelles traitées)
  Simulation 040 : 5832 opérations (100 parcelles traitées)
  Simulation 050 : 6084 opérations (100 parcelles traitées)
  Simulation 060 : 5940 opérations (100 parcelles traitées)
  Simulation 070 : 5532 opérations (100 parcelles traitées)
  Simulation 080 : 6192 opérations (100 parcelles traitées)
  Simulation 090 : 5988 opérations (100 parcelles traitées)
  Simulation 099 : 5748 opérations (100 parcelles traitées)

Fichiers écrits dans : /Users/benjamin/Workspace_GAMA/MAELIA/includes/terrainSA/modeleAgricole/agriculteurs/variants_SMT


In [5]:
df.head(20)

,Trait,nom_MAELIA,plant_sem,annee_deb,id_operation,ordre_op,DATE,PROF,TYPE,DOSE
0,beauce_5_sa_000,P1_2019,ble,2019,SEMIS,,04/10/2019,,,
1,beauce_5_sa_000,P1_2019,ble,2019,RECOLTE,,21/04/2020,,,
2,beauce_5_sa_000,P1_2020,ble,2020,SEMIS,,03/10/2020,,,
3,beauce_5_sa_000,P1_2020,ble,2020,RECOLTE,,21/04/2021,,,
4,beauce_5_sa_000,P1_2021,ble,2021,SEMIS,,04/10/2021,,,
5,beauce_5_sa_000,P1_2021,ble,2021,RECOLTE,,22/04/2022,,,
6,beauce_5_sa_000,P1_2022,ble,2022,SEMIS,,04/10/2022,,,
7,beauce_5_sa_000,P1_2022,ble,2022,RECOLTE,,22/04/2023,,,
8,beauce_5_sa_000,P1_2023,ble,2023,SEMIS,,04/10/2023,,,
9,beauce_5_sa_000,P1_2023,ble,2023,RECOLTE,,21/04/2024,,,


## 4. Lancement des simulations GAMA (headless)

Chaque simulation :
1. Génère un fichier XML de plan d'expérience GAMA.
2. Lance `gama-headless.sh` avec ce XML.
3. Écrit les logs dans `/tmp/gama_smt.log` sans les streamer dans la cellule.

> Pour suivre les logs depuis un terminal : `tail -f /tmp/gama_smt.log`


In [6]:
# ─── Générateur XML headless ──────────────────────────────────────────────────

def make_xml(nom_simu, datedose_path):
    """Génère le contenu XML du plan d'expérience GAMA headless."""
    root = ET.Element('Experiment_plan')
    sim  = ET.SubElement(root, 'Simulation')
    sim.set('experiment',  'simulationBase')
    sim.set('finalStep',   str(FINAL_STEP))
    sim.set('id',          '0')
    sim.set('seed',        '1.0')
    sim.set('sourcePath',  str(MODEL_PATH))

    params = ET.SubElement(sim, 'Parameters')

    def p(name, ptype, value, var):
        el = ET.SubElement(params, 'Parameter')
        el.set('name', name)
        el.set('type', ptype)
        el.set('value', str(value))
        el.set('var', var)

    p('executerSurCluster: ',                  'BOOLEAN', 'false',                               'executerSurCluster')
    p('cheminRacineMaelia',                    'STRING',  CHEMIN_RACINE,                         'cheminRacineMaelia')
    p('cheminModeleVersDonnees',               'STRING',  str(MAELIA_ROOT / 'includes') + '/',   'cheminModeleVersDonnees')
    p('cheminSorties',                         'STRING',  str(LOG_DIR),                          'cheminRelatifDuDossierDeSortieDeSimulation')
    p('anneeDebutSimulation : ',               'INT',     ANNEE_DEBUT,                           'anneeDebutSimulation')
    p('nbAnneesSimulation : ',                 'INT',     NB_ANNEES,                             'nbAnneesSimulation')
    p('nomSimulation : ',                      'STRING',  nom_simu,                              'nomSimulation')
    p('nomDecoupageZonePourLectureFichiers : ', 'STRING',  NOM_DECOUPAGE,                         'nomDecoupageZonePourLectureFichiers')
    p('simulationSurParcelle : ',              'BOOLEAN', 'false',                               'executerUneSeuleParcelle')
    p('idParcelleASimuler : ',                 'STRING',  ALL_PARCELLES[0],                      'nomParcelleAffichee')
    p('reajusterSurfaceParIlot : ',            'BOOLEAN', 'false',                               'reajusterSurfaceParIlot')
    p('executerModeleAgricole : ',             'BOOLEAN', 'true',                                'executerModeleAgricole')
    p('nomChoixAssolement : ',                 'STRING',  'DateDose',                            'nomChoixAssolement')
    p('fichierDateDose : ',                    'STRING',  str(datedose_path),                    'fichierDateDose')
    p('avecContrainteDeMainOeuvre : ',         'BOOLEAN', 'false',                               'avecContrainteDeMainOeuvre')
    p('isIrrigationSimulee : ',                'BOOLEAN', 'false',                               'isIrrigationSimulee')
    p('executerModeleHydrographique : ',       'BOOLEAN', 'false',                               'executerModeleHydrographique')
    p('executerModeleNormatif : ',             'BOOLEAN', 'false',                               'executerModeleNormatif')
    p('executerEcritureFichiers : ',           'BOOLEAN', 'true',                                'executerEcritureFichiers')
    p('sorties eau',                           'BOOLEAN', 'true',                                'sorties_eau')
    p('sorties azote',                         'BOOLEAN', 'true',                                'sorties_azote')
    p('sorties carbone et GES',                'BOOLEAN', 'true',                                'sorties_carboneGES')
    p('Sortie N_Cstock_Parcelles',             'BOOLEAN', 'true',                                'N_Cstock_Parcelles')
    p('Sortie N_lixi_Parcelles',               'BOOLEAN', 'true',                                'N_lixi_Parcelles')
    # Note: 'suiviOTParParcelle' est déjà true par défaut dans launcherBase.gaml.
    # Son label GAML contient des accents ('Suivi détaillé…') qui ne peuvent pas
    # être encodés sans risque dans le XML headless → paramètre omis volontairement.

    ET.SubElement(sim, 'Outputs')

    try:
        ET.indent(root, space='  ')
    except AttributeError:
        pass  # Python < 3.9

    buf = io.BytesIO()
    ET.ElementTree(root).write(buf, encoding='UTF-8', xml_declaration=True)
    xml_str = buf.getvalue().decode('UTF-8')
    # Normaliser la déclaration XML (double quotes)
    xml_str = xml_str.replace("<?xml version='1.0' encoding='UTF-8'?>",
                              '<?xml version="1.0" encoding="UTF-8" standalone="no"?>')
    return xml_str


# ─── Fonction de lancement + écriture des logs ───────────────────────────────

def find_output_dir(nom_simu, ts_before):
    """Trouve le répertoire de sortie MAELIA créé après ts_before."""
    expected_prefix = f'{NOM_DECOUPAGE}_{nom_simu}_'
    for _ in range(20):
        matches = [
            p for p in LOG_DIR.glob(f'{NOM_DECOUPAGE}_*')
            if p.is_dir()
            and p.stat().st_mtime >= ts_before
            and (p.name.startswith(expected_prefix) or nom_simu in p.name)
        ]
        if matches:
            return sorted(matches, key=lambda p: p.stat().st_mtime)[-1]
        time.sleep(1)
    return None


def run_simulation(nom_simu, xml_path, gama_ws):
    """
    Lance une simulation GAMA headless.
    Écrit stdout/stderr dans GAMA_LOG sans streamer dans la cellule notebook.
    Retourne (code_retour, out_dir | None).
    """
    gama_ws.mkdir(parents=True, exist_ok=True)
    cmd = ['bash', GAMA_HEADLESS, str(xml_path), str(gama_ws)]
    t0  = time.time()
    ts_before = t0 - 0.5

    print(f'\n{nom_simu} lancé. Logs terminal : tail -f {GAMA_LOG}', flush=True)

    with open(GAMA_LOG, 'a') as flog:
        flog.write(f'\n{"=" * 62}\n[{nom_simu}] {time.strftime("%H:%M:%S")}\n')
        flog.write(f'Commande: {" ".join(cmd)}\n{"=" * 62}\n')
        proc = subprocess.run(
            cmd,
            stdout=flog,
            stderr=subprocess.STDOUT,
            text=True,
        )

    elapsed  = time.time() - t0
    ok       = (proc.returncode == 0)
    out_dir  = find_output_dir(nom_simu, ts_before) if ok else None
    status   = 'OK' if ok else f'ERREUR (code {proc.returncode})'

    print(f'  [{status}] {nom_simu} terminé en {elapsed:.1f}s', flush=True)
    if out_dir:
        print(f'  Sorties : {out_dir.name}', flush=True)
    else:
        print(f'  Sorties : non trouvées sous {LOG_DIR}', flush=True)
    return proc.returncode, out_dir


In [7]:
# ─── Boucle principale ────────────────────────────────────────────────────────

# Vider le fichier de log
GAMA_LOG.write_text('')
print(f'Log GAMA → {GAMA_LOG}')
print('Pour suivre en temps réel dans un terminal :')
print(f'  tail -f {GAMA_LOG}')
print()

results = {}    # nom_simu → Path | None
t_global = time.time()

for i in range(N_SIMU):
    nom_simu  = f'smt_{i:03d}'
    csv_path  = VARIANTS_DIR / f'dateDose_{nom_simu}.csv'
    xml_path  = XML_DIR / f'{nom_simu}.xml'
    gama_ws   = XML_DIR / f'ws_{nom_simu}'

    # Écrire le XML
    xml_path.write_text(make_xml(nom_simu, csv_path))

    # Lancer la simulation
    ret, out_dir = run_simulation(nom_simu, xml_path, gama_ws)
    results[nom_simu] = out_dir

n_ok    = sum(1 for v in results.values() if v is not None)
elapsed = time.time() - t_global
print(f'\n{"═" * 62}')
print(f'  Terminé : {n_ok}/{N_SIMU} simulations OK  ({elapsed:.0f}s total)')
print(f'{"═" * 62}')


Log GAMA → /tmp/gama_smt_terrainSA.log
Pour suivre en temps réel dans un terminal :
  tail -f /tmp/gama_smt_terrainSA.log


smt_000 lancé. Logs terminal : tail -f /tmp/gama_smt_terrainSA.log
  [OK] smt_000 terminé en 127.0s
  Sorties : terrainSA_smt_000_2026-05-12_10_11_02

smt_001 lancé. Logs terminal : tail -f /tmp/gama_smt_terrainSA.log
  [OK] smt_001 terminé en 123.3s
  Sorties : terrainSA_smt_001_2026-05-12_10_13_10

smt_002 lancé. Logs terminal : tail -f /tmp/gama_smt_terrainSA.log
  [OK] smt_002 terminé en 119.2s
  Sorties : terrainSA_smt_002_2026-05-12_10_15_11

smt_003 lancé. Logs terminal : tail -f /tmp/gama_smt_terrainSA.log
  [OK] smt_003 terminé en 121.4s
  Sorties : terrainSA_smt_003_2026-05-12_10_17_13

smt_004 lancé. Logs terminal : tail -f /tmp/gama_smt_terrainSA.log
  [OK] smt_004 terminé en 116.7s
  Sorties : terrainSA_smt_004_2026-05-12_10_19_11

smt_005 lancé. Logs terminal : tail -f /tmp/gama_smt_terrainSA.log
  [OK] smt_005 terminé en 117.9s
  Sorties : terrainSA

In [8]:
# ─── Reconstruction des résultats depuis le répertoire de logs ────────────────
# Exécuter cette cellule si les simulations ont déjà tourné mais que `results`
# n'est plus en mémoire (kernel redémarré, interruption, etc.).

def _output_dir_has_expected_files(path):
    return (
        (path / 'sorties_CN.csv').exists()
        or (path / 'suiviOTParParcelle.csv').exists()
        or (path / 'suiviOTParParcelle_journalier.csv').exists()
    )

def _batch_start_cutoff():
    times = []
    for i in range(N_SIMU):
        console = XML_DIR / f'ws_smt_{i:03d}' / 'console-outputs-0.txt'
        if console.exists():
            times.append(console.stat().st_mtime)
    return min(times) - 120 if times else 0

def rebuild_results_from_logs():
    latest = {}
    unmatched = []
    cutoff = _batch_start_cutoff()

    for d in LOG_DIR.glob(f'{NOM_DECOUPAGE}_*'):
        if not d.is_dir() or not _output_dir_has_expected_files(d):
            continue
        if 'smoke_patch' in d.name or d.stat().st_mtime < cutoff:
            continue

        matched = False
        for i in range(N_SIMU):
            nom_simu = f'smt_{i:03d}'
            if f'_{nom_simu}_' in d.name or d.name.startswith(f'{NOM_DECOUPAGE}_{nom_simu}_'):
                if nom_simu not in latest or d.stat().st_mtime > latest[nom_simu].stat().st_mtime:
                    latest[nom_simu] = d
                matched = True
                break

        if not matched:
            unmatched.append(d)

    # Compatibility with logs produced before the launcher fix, when all folders
    # could be named terrainTest_beauce_5_* instead of terrainTest_smt_XXX_*.
    missing = [f'smt_{i:03d}' for i in range(N_SIMU) if f'smt_{i:03d}' not in latest]
    for nom_simu, d in zip(missing, sorted(unmatched, key=lambda p: p.stat().st_mtime)):
        latest[nom_simu] = d

    return {k: v for k, v in sorted(latest.items())}

results = rebuild_results_from_logs()

n_found = len(results)
print(f'Dossiers trouvés : {n_found}/{N_SIMU}')
for nom, d in list(results.items())[:10]:
    print(f'  {nom} → {d.name}')
if n_found > 10:
    print(f'  … ({n_found - 10} autres)')
if n_found == 0:
    print(f'Aucun dossier exploitable trouvé sous {LOG_DIR}')


Dossiers trouvés : 100/100
  smt_000 → terrainSA_smt_000_2026-05-12_10_11_02
  smt_001 → terrainSA_smt_001_2026-05-12_10_13_10
  smt_002 → terrainSA_smt_002_2026-05-12_10_15_11
  smt_003 → terrainSA_smt_003_2026-05-12_10_17_13
  smt_004 → terrainSA_smt_004_2026-05-12_10_19_11
  smt_005 → terrainSA_smt_005_2026-05-12_10_21_08
  smt_006 → terrainSA_smt_006_2026-05-12_10_23_08
  smt_007 → terrainSA_smt_007_2026-05-12_10_25_08
  smt_008 → terrainSA_smt_008_2026-05-12_10_27_06
  smt_009 → terrainSA_smt_009_2026-05-12_10_29_09
  … (90 autres)


## 5. Collecte des résultats

In [10]:
# ─── Diagnostic post-run ─────────────────────────────────────────────────────
print('═' * 62)
print('DIAGNOSTIC')
print('═' * 62)

for i in range(N_SIMU):
    nom_simu = f'smt_{i:03d}'
    gama_ws  = XML_DIR / f'ws_{nom_simu}'
    out_dir  = results.get(nom_simu)

    print(f'\n── {nom_simu} ──')

    # 1. Code retour GAMA
    console = gama_ws / 'console-outputs-0.txt'
    if console.exists():
        lines = console.read_text(errors='replace').splitlines()
        # Dernières lignes utiles (filtrer le bruit)
        tail = [l for l in lines if l.strip() and not l.startswith('Adding unit')][-10:]
        print('  console (10 dernières lignes utiles) :')
        for l in tail:
            print(f'    {l}')
    else:
        print(f'  console-outputs-0.txt introuvable dans {gama_ws}')

    # 2. Dossier de sortie MAELIA trouvé ?
    if out_dir:
        print(f'  → dossier sortie : {out_dir}')
        csvs = list(out_dir.glob('*.csv'))
        print(f'  → fichiers CSV   : {[c.name for c in csvs]}')
    else:
        print(f'  → aucun dossier sortie trouvé sous {LOG_DIR}')
        # Cherche quand même des dossiers récents pour aider au diagnostic
        all_dirs = sorted(LOG_DIR.glob(f'{NOM_DECOUPAGE}_*'), key=lambda p: p.stat().st_mtime, reverse=True)
        if all_dirs:
            print(f'  → dossiers existants les plus récents dans log/ :')
            for d in all_dirs[:5]:
                print(f'      {d.name}')
        else:
            print(f'  → log/ est vide ou n\'existe pas')


══════════════════════════════════════════════════════════════
DIAGNOSTIC
══════════════════════════════════════════════════════════════

── smt_000 ──
  console (10 dernières lignes utiles) :
    1_sINIT
    Image Display Surface created for simulation Simulation 0
    [LayeredDisplayOutput.update] START for Meteo thread=pool-1-thread-1
    [LayeredDisplayOutput.update] after getData().update=1ms
    [LayeredDisplayOutput.update] END for Meteo total=1ms
    Image Display Surface created for simulation Simulation 0
    [LayeredDisplayOutput.update] START for map thread=pool-1-thread-1
    [LayeredDisplayOutput.update] after getData().update=0ms
    [LayeredDisplayOutput.update] END for map total=0ms
    ....................................................................................................> GAMA  : Simulation running                            for: _________ 110013ms
  → dossier sortie : /Users/benjamin/Workspace_GAMA/MAELIA/models/main/log/terrainSA_smt_000_2026-05-12_10_

In [11]:
import struct as _struct, glob as _glob

# --- Lecture DBF ilots (une seule fois) ---
def _read_dbf_ilot_sol(dbf_path):
    """Renvoie {ilot_name: ID_SOL} depuis le shapefile ilots.dbf."""
    with open(dbf_path, 'rb') as _f:
        _d = _f.read()
    _n   = _struct.unpack('<I', _d[4:8])[0]
    _hs  = _struct.unpack('<H', _d[8:10])[0]
    _rs  = _struct.unpack('<H', _d[10:12])[0]
    _flds = []
    _off = 32
    while _d[_off] != 0x0D:
        _name = _d[_off:_off+11].split(b'\x00')[0].decode('latin1')
        _flds.append((_name, _d[_off+16]))
        _off += 32
    _off = _hs
    _out = {}
    for _ in range(_n):
        if _d[_off] == 0x2A:
            _off += _rs
            continue
        _off += 1
        _row = {}
        for _nm, _ln in _flds:
            _row[_nm] = _d[_off:_off+_ln].decode('latin1').strip()
            _off += _ln
        _out[_row['ID_ILOT']] = _row['ID_SOL']
    return _out

_DBF_PATH = (MAELIA_ROOT / 'includes' / NOM_DECOUPAGE / 'modeleAgricole/ilots/dansZone/ilots.dbf')
_ILOT_SOL = _read_dbf_ilot_sol(_DBF_PATH)   # {ilot_name -> sol_type}


def _read_corr_ilot_meteo(out_dir):
    """Renvoie {ilot_name: zone_meteo_int} depuis corresponsanceIlotZoneMeteo.csv."""
    _p = out_dir / 'corresponsanceIlotZoneMeteo.csv'
    if not _p.exists():
        return {}
    _df = pd.read_csv(_p, sep=';')
    return dict(zip(_df['ilot'], _df['zoneMeteo'].astype(int)))


SOL_TYPES   = ['18960_27491', '330104_330367_1', '330151_330489_1']  # A, B, C
METEO_CODES = [4756, 13218, 13761]  # sudouest, beauce, oceanique


def _zone(parcelle):
    for z in ('beauce', 'oceanique', 'sudouest'):
        if z in parcelle:
            return z
    return 'autre'


def read_simulation_outputs(out_dir):
    """
    Retourne un DataFrame long (parcelle x annee) avec les 3 sorties cibles.
    Colonnes : annee, parcelle, N_lixi, dCorg, rdt, zone, sol_type, zone_meteo
    """
    f_cn = out_dir / 'sorties_CN.csv'
    if not f_cn.exists():
        return pd.DataFrame()
    df = pd.read_csv(f_cn, sep=';',
                     usecols=['annee', 'parcelle', 'couvert',
                              'N_lixivie[kgN/ha]', 'delta_Corg[kgC/ha]'])
    df = df[df['couvert'].notna() & (df['couvert'] != 'solnu')].drop(columns='couvert')
    df = df.rename(columns={'N_lixivie[kgN/ha]': 'N_lixi',
                             'delta_Corg[kgC/ha]': 'dCorg'})

    f_ot = out_dir / 'suiviOTParParcelle.csv'
    if f_ot.exists():
        df_ot = pd.read_csv(f_ot, sep=';',
                            usecols=['annee', 'parcelle', 'OT',
                                     'RECOLTE_rendement[t/ha]'])
        df_ot = (df_ot[df_ot['OT'] == 'RECOLTE']
                 .drop(columns='OT')
                 .rename(columns={'RECOLTE_rendement[t/ha]': 'rdt'}))
        df = df.merge(df_ot, on=['annee', 'parcelle'], how='left')

    df['zone'] = df['parcelle'].apply(_zone)

    # sol_type et zone_meteo depuis shapefile + corresponsance
    _corr = _read_corr_ilot_meteo(out_dir)
    def _ilot_from_parcelle(p):
        if str(p).startswith('beauce_5_sa_'):
            return 'beauce_5'
        return '_'.join(str(p).rsplit('_', 1)[:-1])
    def _sol_type(p):
        return _ILOT_SOL.get(_ilot_from_parcelle(p), None)
    def _zone_meteo(p):
        return _corr.get(_ilot_from_parcelle(p), None)

    df['sol_type']   = df['parcelle'].apply(_sol_type)
    df['zone_meteo'] = df['parcelle'].apply(_zone_meteo)
    return df


# --- Collecte ---
all_data     = {}
summary_rows = []

for nom_simu, out_dir in results.items():
    if not (out_dir and out_dir.exists()):
        summary_rows.append({'simulation': nom_simu, 'ok': False})
        continue
    df = read_simulation_outputs(out_dir)
    if df.empty:
        summary_rows.append({'simulation': nom_simu, 'ok': False})
        continue

    all_data[nom_simu] = df
    row = {'simulation': nom_simu, 'ok': True,
           'n_parcelles': df['parcelle'].nunique()}
    for m in ('N_lixi', 'dCorg', 'rdt'):
        if m in df.columns:
            row[f'{m}_mean'] = df[m].mean()
            row[f'{m}_std']  = df[m].std()
    summary_rows.append(row)

results_df = pd.DataFrame(summary_rows)

print(f'Resultats : {results_df["ok"].sum()} simulations OK')
mean_cols = [c for c in results_df.columns if c.endswith('_mean')]
display(results_df[['simulation', 'ok', 'n_parcelles'] + mean_cols].round(3))

print('\nDistribution (zone, sol_type, zone_meteo) sur smt_000 :')
_s0 = all_data.get('smt_000', pd.DataFrame())
if not _s0.empty:
    _avg = _s0.groupby(['zone', 'sol_type', 'zone_meteo']).size().reset_index(name='n_parcelles')
    display(_avg)

# --- Sauvegarde detail par simulation ---
results_df.to_csv(XML_DIR / 'resultats_smt.csv', index=False)
for nom_simu, df in all_data.items():
    df.to_csv(XML_DIR / f'detail_{nom_simu}.csv', index=False)
print(f'\nFichiers sauvegardes dans {XML_DIR}/')


Resultats : 100 simulations OK


,simulation,ok,n_parcelles,N_lixi_mean,dCorg_mean,rdt_mean
0,smt_000,True,100,3.732,-227.580,4.135
1,smt_001,True,100,3.807,-229.885,4.118
2,smt_002,True,100,3.669,-212.301,4.131
3,smt_003,True,100,3.679,-211.804,4.088
4,smt_004,True,100,3.783,-226.989,4.137
...,...,...,...,...,...,...
95,smt_095,True,100,3.891,-244.494,4.163
96,smt_096,True,100,3.652,-211.680,4.132
97,smt_097,True,100,3.730,-219.824,4.124
98,smt_098,True,100,3.625,-213.730,4.128



Distribution (zone, sol_type, zone_meteo) sur smt_000 :


,zone,sol_type,zone_meteo,n_parcelles
0,beauce,18960_27491,13218,500



Fichiers sauvegardes dans /tmp/maelia_smt_terrainSA_xml/


In [12]:
# --- 8. Export for Metamodeling ---
import pandas as pd
import numpy as np

print("Collecte des résultats et fusion avec le plan d'expérience (13,000 points)...")

_records = []
parc_to_idx = {p: k for k, p in enumerate(ALL_PARCELLES)}

for nom_simu, out_dir in results.items():
    if not (out_dir and out_dir.exists()): continue
    sim_idx = int(nom_simu.split('_')[-1])
    
    df = read_simulation_outputs(out_dir)
    if df.empty: continue
    
    # Mapper chaque parcelle à son point DOE spécifique
    df['point_idx'] = sim_idx * len(ALL_PARCELLES) + df['parcelle'].map(parc_to_idx)
    
    # Calculer la moyenne par parcelle (sur les années de simulation) pour N_lixi, dCorg, rdt
    df_parc = df.groupby(['parcelle', 'point_idx', 'zone', 'sol_type']).agg({
        'N_lixi': 'mean', 'dCorg': 'mean', 'rdt': 'mean'
    }).reset_index()
    
    _records.append(df_parc)

if _records:
    df_full = pd.concat(_records, ignore_index=True)
    
    # Ajouter les caractéristiques agricoles (x0...x25) depuis xt
    # xt contient N_DOE points
    point_indices = df_full['point_idx'].values
    xt_subset = xt[point_indices]
    
    for j in range(xt.shape[1]):
        df_full[f'feat_{j}'] = xt_subset[:, j]
    
    # Ajout des milieux agrégés (Climat / Sol)
    zone_map = {z: i for i, z in enumerate(sorted(df_full['zone'].unique()))}
    soil_map = {s: i for i, s in enumerate(sorted(df_full['sol_type'].unique()))}
    
    df_full['Milieu (Climat)'] = df_full['zone'].map(zone_map)
    df_full['Milieu (Sol)']    = df_full['sol_type'].map(soil_map)
    
    # Réorganiser les colonnes pour les métamodèles (26 agri + 2 env + 3 outputs)
    X_agri_names = [f'feat_{j}' for j in range(26)]
    export_cols = X_agri_names + ['Milieu (Climat)', 'Milieu (Sol)', 'N_lixi', 'dCorg', 'rdt', 'point_idx', 'parcelle']
    
    df_export = df_full[export_cols]
    export_path = XML_DIR / 'dataset_metamodel.csv'
    df_export.to_csv(export_path, index=False)
    
    print(f"Export réussi : {export_path}")
    print(f"Nombre de points : {len(df_export)}")
    print(f"Structure : 26 agri + 2 env + 3 sorties")
else:
    print("Aucune donnée collectée.")


Collecte des résultats et fusion avec le plan d'expérience (13,000 points)...
Export réussi : /tmp/maelia_smt_terrainSA_xml/dataset_metamodel.csv
Nombre de points : 10000
Structure : 26 agri + 2 env + 3 sorties
